In [1]:
import pandas as pd 

In [8]:
evaluation_df = pd.read_csv("../reports/model_test_predictions.csv")
evaluation_df.head()

,id,actual,logistic_prediction,svm_prediction
0,1047,2,2,2
1,1373,2,2,2
2,1864,2,2,2
3,1738,0,1,1
4,255,2,1,1


In [9]:
evaluation_df.shape

(376, 4)

There are 4 columns:
- id
- actual
- logistic_prediction
- svm_prediction

The idea here is to implement an Error Analysis to check these 4 situations:
- both models correct
- only LR correct
- only SVM correct
- both models incorrect

What is considered correct is when a predicted class is equal to the actual class

In [11]:
evaluation_df["lr_correct"] = evaluation_df["actual"] == evaluation_df["logistic_prediction"]
evaluation_df["svm_correct"] = evaluation_df["actual"] == evaluation_df["svm_prediction"]


In [17]:
lr_correct_count = evaluation_df["lr_correct"].sum()
print(lr_correct_count)

261


In [18]:
svm_correct_count = evaluation_df["svm_correct"].sum()
print(svm_correct_count)

260


In [22]:
evaluation_df["both_correct"] = (evaluation_df["lr_correct"] & evaluation_df["svm_correct"])

both_correct_count = evaluation_df["both_correct"].sum()
print(both_correct_count)


250


In [25]:
evaluation_df["only_lr_correct"] = (evaluation_df["lr_correct"] & ~evaluation_df["svm_correct"])
only_lr_correct_count = evaluation_df["only_lr_correct"].sum()
print(only_lr_correct_count)

11


In [26]:
evaluation_df["only_svm_correct"] = (~evaluation_df["lr_correct"] & evaluation_df["svm_correct"])
only_svm_correct_count = evaluation_df["only_svm_correct"].sum()
print(only_svm_correct_count)

10


In [27]:
evaluation_df["both_incorrect"] = (~evaluation_df["lr_correct"] & ~evaluation_df["svm_correct"])
both_incorrect_count = evaluation_df["both_incorrect"].sum()
print(both_incorrect_count)

105


In [31]:
error_summary_df = pd.DataFrame({
    "Outcome": ["Both Correct", "Only LR Correct", "Only SVM Correct", "Both Incorrect"],
    "Count": [both_correct_count, only_lr_correct_count, only_svm_correct_count, both_incorrect_count]
})

display(error_summary_df)

,Outcome,Count
0,Both Correct,250
1,Only LR Correct,11
2,Only SVM Correct,10
3,Both Incorrect,105


Both models correctly classified 250 out of 376 respondents, while both incorrectly classified 105 respondents.

The models perform very similarly. Logistic Regression correctly classifies just one more respondent than SVM.

They don't always make the same mistakes. There are 21 respondents where one model is correct and the other is wrong.



In [28]:
evaluation_df["actual"].value_counts()

actual
2    198
1     95
0     83
Name: count, dtype: int64

Class 2 has more than twice as many respondents compared to the other classes => class imbalance

In [29]:
evaluation_df.groupby("actual")["lr_correct"].sum()

actual
0     48
1     33
2    180
Name: lr_correct, dtype: int64

In [ ]:
evaluation_df.groupby("actual")["lr_correct"].mean()    #recall

actual
0    0.578313
1    0.347368
2    0.909091
Name: lr_correct, dtype: float64

In [33]:
evaluation_df.groupby("actual")["svm_correct"].sum()

actual
0     46
1     40
2    174
Name: svm_correct, dtype: int64

In [36]:
evaluation_df.groupby("actual")["svm_correct"].mean()    #recall

actual
0    0.554217
1    0.421053
2    0.878788
Name: svm_correct, dtype: float64

In [47]:
class_recall_df = (
    evaluation_df.groupby("actual")[["lr_correct","svm_correct"]].mean() * 100
)

display(class_recall_df.round(2))

,lr_correct,svm_correct
actual,,
0,57.83,55.42
1,34.74,42.11
2,90.91,87.88


In [57]:
class_recall_df.rename(columns={
    "lr_correct": "LR Recall (%)",
    "svm_correct": "SVM Recall (%)"
})

,LR Recall (%),SVM Recall (%)
actual,,
0,57.83,55.42
1,34.74,42.11
2,90.91,87.88


LR has a higher recall (better performance) for two classes (0 and 2)

SVM has a higher recall for class 1,  which happens to also be the same class where both models struggle the most

In [59]:
lr_class1_errors_df = evaluation_df[(evaluation_df["actual"] == 1) & (~evaluation_df["lr_correct"])]

lr_class1_errors_df.shape

(62, 10)

This confirms that Logistic Regression misclassified 62 of the 95 former users

In [60]:
lr_class1_errors_df["logistic_prediction"].value_counts()

logistic_prediction
2    35
0    27
Name: count, dtype: int64

out of the 62 wrong predictions 35 was predicted as recent user and 27 as never used

In [61]:
svm_class1_errors_df = evaluation_df[(evaluation_df["actual"] == 1) & (~evaluation_df["svm_correct"])]

svm_class1_errors_df.shape

(55, 10)

In [62]:
svm_class1_errors_df["svm_prediction"].value_counts()

svm_prediction
2    32
0    23
Name: count, dtype: int64

out of the 55 wrong predictions 32 was predicted as recent users and 23 as never used

Both models misclassify former users as recent users slightly more often than as never users

In [63]:
evaluation_df.groupby("actual")["both_incorrect"].sum()

actual
0    33
1    55
2    17
Name: both_incorrect, dtype: int64

Class 0: Both models misclassified 33 never users.

Class 1: Both models misclassified 55 former users.

Class 2: Both models misclassified 17 recent users.

In [68]:
shared_error_df = evaluation_df.groupby("actual")["both_incorrect"].mean()*100
shared_error_df = shared_error_df.to_frame()

shared_error_df = shared_error_df.rename(
    columns={"both_incorrect": "Shared Error Rate (%)"}
).round(2)

display(shared_error_df)

,Shared Error Rate (%)
actual,
0,39.76
1,57.89
2,8.59
